## Import libraries

In [ ]:
# harp-data for data interface, pandas for dataframe manipulation, matplotlib for plotting
from harp import data
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

## Load device reader

In [ ]:
# open_dataset() finds the device.yml inside the dataset folder and returns a reader for the recorded registers
reader = data.open_dataset("Data/Hobgoblin.harp")

## Load register data

In [ ]:
# The data for each register can be loaded into a pandas dataframe by simply calling read() with the register name
# Bitmask registers are read as a single integer column by default, demux_bit_masks=True gives one boolean column per pin instead
# For a full list of registers, refer to the API documentation at https://harp-tech.org/api/Harp.Hobgoblin.html
digital_output_set = reader.read("DigitalOutputSet", demux_bit_masks=True)
digital_output_clear = reader.read("DigitalOutputClear", demux_bit_masks=True)
digital_input_state = reader.read("DigitalInputState", demux_bit_masks=True)

## Inspect dataframes

In [ ]:
print("LED onset time")
print(digital_output_set.head())
print("\nLED offset time")
print(digital_output_clear.head())
print("\nButton press time")
print(digital_input_state.head())

## Preprocess dataframes

In [ ]:
# Discard unused channels
digital_output_set = digital_output_set["GP15"]
digital_output_clear = digital_output_clear["GP15"]
digital_input_state = digital_input_state["GP2"]

# Keep button presses
digital_input_state = digital_input_state[digital_input_state == True]

## Plot session

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

# Extract times
press_times = digital_input_state.index

# Plot LED epochs as shaded blue regions
for start, end in zip(digital_output_set.index, digital_output_clear.index):
    ax.axvspan(start, end, alpha=0.3)

# Plot button press as black lines
ax.vlines(digital_input_state.index, ymin=0.5, ymax=1, color = 'black')

# Cleanup plot
ax.set_ylim(0, 1.2)
ax.set_yticks([])
ax.set_xlabel("Time (s)")
ax.set_title("Reaction Time Plot")

plt.show()

## Calculate hit response time

In [ ]:
# Keep only 1st button press that occurs within response_window/timeout
valid_response_times = []
for trial_number in range(len(digital_output_set.index)):
    led_on = digital_output_set.index[trial_number]
    response_window = digital_output_clear.index[trial_number] - digital_output_set.index[trial_number]
    for button_press in digital_input_state.index:
        response_time = button_press - led_on
        if 0 < response_time < response_window:
            valid_response_times.append(response_time)
            break

## Calculate hit/miss percentage

In [ ]:
num_valid_responses = len(valid_response_times)
num_total_trials = len(digital_output_set.index)
hit_percentage = num_valid_responses / num_total_trials * 100
print(f"There were {num_valid_responses} valid responses out of {num_total_trials} trials, giving a hit rate of {hit_percentage}%")

## Plot hit response times

In [ ]:
pd.DataFrame(valid_response_times).boxplot(grid = False)

In [ ]:
print(np.average(valid_response_times))